In [25]:
import pandas as pd
from pyprojroot import here
from scipy import stats
from statsmodels.stats.multitest import multipletests
import numpy as np
import json

In [26]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"
PROCESSED_DATA_DIR = here() / "data" / "processed"
RESULTS_DATA_DIR = here() / "results" 

In [27]:
database = "crime_rates_AB"
df = pd.read_csv(INTERIM_DATA_DIR/f"{database}.csv")

### We perform a Wilcoxon test with Benjamini-Hochberg correction using 2025 as comparison base

In [28]:
base = df[df['year'] == 2025][['CVEGEO','crime_rate']].rename(columns={'crime_rate':'rate_2025'})

years = sorted([y for y in df['year'].unique() if y != 2025])

results = []
plot_payload = {}

In [29]:
arr_2025 = base['rate_2025'].dropna().values.astype(float)
p99_2025 = np.percentile(arr_2025, 99)
plot_payload['2025'] = {
    'raw':  arr_2025[arr_2025 <= p99_2025].tolist(),
    'q0':   float(np.percentile(arr_2025, 0)),
    'q25':  float(np.percentile(arr_2025, 25)),
    'q50':  float(np.percentile(arr_2025, 50)),
    'q75':  float(np.percentile(arr_2025, 75)),
    'q99':  float(p99_2025),
    'mean': float(np.mean(arr_2025)),
}

In [30]:
for yr in years:
    yr_df  = df[df['year'] == yr][['CVEGEO','crime_rate']].rename(columns={'crime_rate': 'rate_yr'})
    merged = base.merge(yr_df, on='CVEGEO', how='left')
    merged['rate_yr'] = merged['rate_yr'].fillna(0.0)

    r2025 = merged['rate_2025'].values.astype(float)
    ryr   = merged['rate_yr'].values.astype(float)

    # Drop pairs where 2025 is NaN
    mask  = ~np.isnan(r2025)
    r2025_clean, ryr_clean = r2025[mask], ryr[mask]

    med_diff = float(np.nanmedian(r2025_clean - ryr_clean))
    result   = stats.wilcoxon(r2025_clean, ryr_clean, alternative='two-sided', nan_policy='omit')

    results.append({
        'year':        yr,
        'n':           int(mask.sum()),
        'W':           float(result.statistic),
        'p_raw':       float(result.pvalue),
        'median_diff': round(med_diff, 4),
    })

    p99 = np.percentile(ryr_clean, 99)
    plot_payload[str(yr)] = {
        'raw':  ryr_clean[ryr_clean <= p99].tolist(),
        'q0':   float(np.percentile(ryr_clean, 0)),
        'q25':  float(np.percentile(ryr_clean, 25)),
        'q50':  float(np.percentile(ryr_clean, 50)),
        'q75':  float(np.percentile(ryr_clean, 75)),
        'q99':  float(p99),
        'mean': float(np.mean(ryr_clean)),
    }

In [31]:
# Benjamini-Hochberg correction
res_df = pd.DataFrame(results)
reject, p_adj, _, _ = multipletests(res_df['p_raw'], method='fdr_bh')
res_df['p_bh']        = [float(x) for x in p_adj]
res_df['significant'] = [bool(x)  for x in reject]

print(res_df.to_string(index=False))

 year    n         W        p_raw  median_diff         p_bh  significant
 2015 2456  505022.0 9.266393e-72       2.2871 9.266393e-71         True
 2016 2456  518929.0 3.170940e-69       2.5098 1.585470e-68         True
 2017 2456  641817.0 2.748616e-38       0.0000 9.162052e-38         True
 2018 2456  922335.0 9.812645e-06       0.0000 2.453161e-05         True
 2019 2456 1049280.0 8.842186e-01       0.0000 8.842186e-01        False
 2020 2456  944958.5 9.019398e-04       0.0000 1.503233e-03         True
 2021 2456  963115.0 4.354463e-03       0.0000 6.220661e-03         True
 2022 2456 1027474.5 2.225473e-01       0.0000 2.472748e-01        False
 2023 2456 1012339.0 1.104960e-01       0.0000 1.381200e-01        False
 2024 2456  896714.0 2.713912e-05       0.0000 5.427823e-05         True


In [32]:
with open(PROCESSED_DATA_DIR/'plot_summary.json', 'w') as f:
    json.dump(plot_payload, f)

res_df.to_json(PROCESSED_DATA_DIR/'stats_table.json', orient='records')
res_df.to_csv(RESULTS_DATA_DIR/'stats_table.csv', index=False)